In [1]:
from pathlib import Path

import polars as pl
import yaml

with open("../config.budapest.yaml", "r") as fp:
    config = yaml.safe_load(fp)

In [ ]:
def determine_visitation_network(home_work: pl.DataFrame, res: int) -> pl.DataFrame:
    result = pl.DataFrame()
    for i in range(256):
        temp = pl.read_parquet(
            f"{path}/hexagon_network/{res}/partials/connections_{i:02x}.parquet"
        )

        m = temp.join(hw, on="device_id")
        from_home = m.filter(
            (pl.col("source") == pl.col("home")) & (pl.col("target") != pl.col("work"))
        )
        from_work = m.filter(
            (pl.col("source") == pl.col("work")) & (pl.col("target") != pl.col("home"))
        )
        result = pl.concat([result, from_home])
        result = pl.concat([result, from_work])
    return result.group_by(["device_id", "source", "target"]).agg(pl.sum("weight"))


def aggregate_by_trip(
    visitations: pl.DataFrame, home_work: pl.DataFrame
) -> pl.DataFrame:
    return (
        visitations.join(home_work, on="device_id")
        .group_by(["home", "work", "source", "target"])
        .agg(pl.sum("weight"))
    )


def aggregate_by_device(
    visitations: pl.DataFrame, home_work: pl.DataFrame
) -> pl.DataFrame:
    return (
        visitations.join(hw, on="device_id")
        .group_by(["home", "work", "source", "target"])
        .agg(pl.col("device_id").unique().len().alias("weight"))
    )

In [ ]:
THRESHOLD = 3
VARIANT = "v1"

In [4]:
path = f"{config['dirs']['output']}/{config['city']}"

In [12]:
for res in range(7, 11):
    for by in ["device", "trip"]:
        hw = pl.read_parquet(f"{path}/homes/{VARIANT}/home_work_h3_{res}.parquet")
        hw = hw.rename({f"h3_{res}_home": "home", f"h3_{res}_work": "work"})
        hw = hw.drop_nulls()

        hw = hw.filter(
            (pl.col("count_home") > THRESHOLD) & (pl.col("count_work") > THRESHOLD)
        )

        visitations = determine_visitation_network(hw, res=res)
        aggregated = aggregate_by_trip(visitations, hw)
        if by == "device":
            aggregated = aggregate_by_device(visitations, hw)

        aggregated = aggregated.sort(["home", "work", "source", "target"])
        Path(f"{path}/hexagon_network/{res}/commuting_aware").mkdir(
            exist_ok=True, parents=True
        )
        aggregated.write_csv(f"{path}/hexagon_network/{res}/commuting_aware/{by}.csv")
        aggregated.write_parquet(
            f"{path}/hexagon_network/{res}/commuting_aware/{by}.parquet"
        )